# Gaussian Mixture Models & EM Algorithm from Scratch

**Goal:** Implement GMM with full EM algorithm, BIC/AIC model selection, and understand the connection between K-means and GMM.

---

## Core Intuition

A Gaussian Mixture Model assumes data is generated from $K$ Gaussian components:
$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

where $\pi_k$ are mixing weights ($\sum \pi_k = 1$).

**EM (Expectation-Maximization)** maximizes the log-likelihood by alternating:
1. **E-step:** Compute responsibilities (soft assignments): $\gamma_{ik} = P(z_i = k | x_i, \theta)$
2. **M-step:** Update parameters using weighted statistics

EM is **coordinate ascent on the ELBO** (Evidence Lower BOund). Each step is guaranteed to increase (or maintain) the log-likelihood.

**Key formulas:**
- E-step: $\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i|\mu_k, \Sigma_k)}{\sum_j \pi_j \mathcal{N}(x_i|\mu_j, \Sigma_j)}$
- M-step: $\mu_k = \frac{\sum_i \gamma_{ik} x_i}{\sum_i \gamma_{ik}}$, $\Sigma_k = \frac{\sum_i \gamma_{ik}(x_i - \mu_k)(x_i - \mu_k)^T}{\sum_i \gamma_{ik}}$, $\pi_k = \frac{1}{n}\sum_i \gamma_{ik}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Multivariate Gaussian PDF (NumPy)

In [ ]:
def multivariate_gaussian_pdf(X, mu, sigma):
    """Compute multivariate Gaussian PDF.
    
    X: (n, d)
    mu: (d,)
    sigma: (d, d)
    Returns: (n,)
    """
    d = X.shape[1]
    diff = X - mu
    
    # Add small regularization for numerical stability
    sigma_reg = sigma + 1e-6 * np.eye(d)
    
    sign, log_det = np.linalg.slogdet(sigma_reg)
    inv_sigma = np.linalg.inv(sigma_reg)
    
    # Mahalanobis distance: (x-mu)^T Sigma^-1 (x-mu)
    mahal = np.sum(diff @ inv_sigma * diff, axis=1)
    
    log_pdf = -0.5 * (d * np.log(2 * np.pi) + log_det + mahal)
    return np.exp(log_pdf)

# Quick test
X_test = np.array([[0, 0], [1, 1], [2, 2]])
mu_test = np.array([1, 1])
sigma_test = np.eye(2)
print("PDF values:", multivariate_gaussian_pdf(X_test, mu_test, sigma_test))

## 2. Full GMM with EM

In [ ]:
class GaussianMixtureModel:
    """Gaussian Mixture Model with EM algorithm."""
    
    def __init__(self, n_components=3, max_iters=100, tol=1e-6, random_state=42):
        self.n_components = n_components
        self.max_iters = max_iters
        self.tol = tol
        self.random_state = random_state
        
        # Parameters
        self.weights_ = None    # (K,) mixing weights
        self.means_ = None      # (K, d) means
        self.covariances_ = None  # (K, d, d) covariances
        self.log_likelihoods_ = []
        self.responsibilities_ = None  # (n, K)
    
    def _initialize(self, X):
        """Initialize parameters using K-means-style initialization."""
        n, d = X.shape
        rng = np.random.RandomState(self.random_state)
        
        # Random initialization
        idx = rng.choice(n, self.n_components, replace=False)
        self.means_ = X[idx].copy()
        self.covariances_ = np.array([np.eye(d) for _ in range(self.n_components)])
        self.weights_ = np.ones(self.n_components) / self.n_components
    
    def _e_step(self, X):
        """E-step: compute responsibilities (posterior P(z=k|x))."""
        n = X.shape[0]
        responsibilities = np.zeros((n, self.n_components))
        
        for k in range(self.n_components):
            responsibilities[:, k] = self.weights_[k] * multivariate_gaussian_pdf(
                X, self.means_[k], self.covariances_[k]
            )
        
        # Normalize (Bayes rule)
        total = responsibilities.sum(axis=1, keepdims=True)
        total = np.maximum(total, 1e-300)  # prevent division by zero
        responsibilities /= total
        
        return responsibilities
    
    def _m_step(self, X, responsibilities):
        """M-step: update parameters using weighted statistics."""
        n, d = X.shape
        
        # Effective number of points per component
        N_k = responsibilities.sum(axis=0)  # (K,)
        
        for k in range(self.n_components):
            if N_k[k] < 1e-10:
                continue
            
            # Update mean
            self.means_[k] = (responsibilities[:, k:k+1].T @ X).ravel() / N_k[k]
            
            # Update covariance
            diff = X - self.means_[k]  # (n, d)
            weighted_diff = diff * responsibilities[:, k:k+1]  # (n, d)
            self.covariances_[k] = (weighted_diff.T @ diff) / N_k[k]
            
            # Add regularization to prevent singularity
            self.covariances_[k] += 1e-6 * np.eye(d)
        
        # Update weights
        self.weights_ = N_k / n
    
    def _log_likelihood(self, X):
        """Compute log-likelihood of data under current model."""
        n = X.shape[0]
        log_probs = np.zeros((n, self.n_components))
        
        for k in range(self.n_components):
            pdf = multivariate_gaussian_pdf(X, self.means_[k], self.covariances_[k])
            log_probs[:, k] = np.log(self.weights_[k] + 1e-300) + np.log(pdf + 1e-300)
        
        # Log-sum-exp for numerical stability
        max_log = np.max(log_probs, axis=1, keepdims=True)
        log_likelihood = np.sum(max_log.ravel() + np.log(np.sum(np.exp(log_probs - max_log), axis=1)))
        return log_likelihood
    
    def fit(self, X):
        self._initialize(X)
        self.log_likelihoods_ = []
        
        for i in range(self.max_iters):
            # E-step
            responsibilities = self._e_step(X)
            
            # M-step
            self._m_step(X, responsibilities)
            
            # Track log-likelihood
            ll = self._log_likelihood(X)
            self.log_likelihoods_.append(ll)
            
            # Check convergence
            if len(self.log_likelihoods_) > 1:
                if abs(self.log_likelihoods_[-1] - self.log_likelihoods_[-2]) < self.tol:
                    break
        
        self.responsibilities_ = responsibilities
        self.labels_ = np.argmax(responsibilities, axis=1)
        self.n_iter_ = i + 1
        return self
    
    def predict(self, X):
        responsibilities = self._e_step(X)
        return np.argmax(responsibilities, axis=1)
    
    def predict_proba(self, X):
        return self._e_step(X)
    
    def score(self, X):
        """Average log-likelihood per sample."""
        return self._log_likelihood(X) / X.shape[0]
    
    def bic(self, X):
        """Bayesian Information Criterion."""
        n, d = X.shape
        # Number of parameters: K means + K covariances + K-1 weights
        n_params = self.n_components * d + self.n_components * d * (d + 1) / 2 + self.n_components - 1
        return -2 * self._log_likelihood(X) + n_params * np.log(n)
    
    def aic(self, X):
        """Akaike Information Criterion."""
        n, d = X.shape
        n_params = self.n_components * d + self.n_components * d * (d + 1) / 2 + self.n_components - 1
        return -2 * self._log_likelihood(X) + 2 * n_params

# Generate data from a mixture of Gaussians
from sklearn.datasets import make_blobs

np.random.seed(42)
X_gmm, y_gmm = make_blobs(n_samples=400, centers=3, cluster_std=[1.0, 1.5, 0.5], random_state=42)

gmm = GaussianMixtureModel(n_components=3, random_state=42)
gmm.fit(X_gmm)

print(f"Converged in {gmm.n_iter_} iterations")
print(f"Final log-likelihood: {gmm.log_likelihoods_[-1]:.2f}")
print(f"BIC: {gmm.bic(X_gmm):.2f}")

## 3. Visualize EM Convergence

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(gmm.log_likelihoods_, 'b-o', linewidth=2, markersize=4)
plt.xlabel('EM Iteration')
plt.ylabel('Log-Likelihood')
plt.title('EM Convergence: Log-Likelihood Increases Monotonically')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Visualize Gaussian Components and Responsibilities

In [ ]:
def draw_ellipse(mean, cov, ax, n_std=2.0, **kwargs):
    """Draw an ellipse representing a 2D Gaussian at n_std standard deviations."""
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width, height = 2 * n_std * np.sqrt(np.maximum(eigenvalues, 0))
    
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']

# Soft assignments (colored by responsibilities)
ax = axes[0]
resp_colors = gmm.responsibilities_ @ np.array([[1, 0, 0], [0, 0, 1], [0, 1, 0]])
resp_colors = np.clip(resp_colors, 0, 1)
ax.scatter(X_gmm[:, 0], X_gmm[:, 1], c=resp_colors, alpha=0.5, s=15)

for k in range(3):
    draw_ellipse(gmm.means_[k], gmm.covariances_[k], ax, n_std=2,
                 fill=False, edgecolor=colors[k], linewidth=2, linestyle='-')
    draw_ellipse(gmm.means_[k], gmm.covariances_[k], ax, n_std=1,
                 fill=False, edgecolor=colors[k], linewidth=2, linestyle='--')
    ax.scatter(*gmm.means_[k], c=colors[k], marker='X', s=200, edgecolors='black', linewidths=2, zorder=5)

ax.set_title('GMM: Soft Assignments (color = responsibility blend)')
ax.grid(True, alpha=0.3)

# Hard assignments
ax = axes[1]
for k in range(3):
    mask = gmm.labels_ == k
    ax.scatter(X_gmm[mask, 0], X_gmm[mask, 1], c=colors[k], alpha=0.5, s=15, label=f'Cluster {k}')
    draw_ellipse(gmm.means_[k], gmm.covariances_[k], ax, n_std=2,
                 fill=False, edgecolor=colors[k], linewidth=2)
    ax.scatter(*gmm.means_[k], c=colors[k], marker='X', s=200, edgecolors='black', linewidths=2, zorder=5)

ax.set_title('GMM: Hard Assignments (argmax responsibility)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Gaussian Mixture Model: Ellipses show 1 and 2 std', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. K-Means (Hard) vs GMM (Soft) Comparison

In [ ]:
# Generate anisotropic data where GMM shines
np.random.seed(42)
n_per = 150
# Elongated cluster
c1 = np.random.randn(n_per, 2) @ np.array([[3, 0], [0, 0.3]]) + [0, 0]
# Round cluster
c2 = np.random.randn(n_per, 2) * 0.8 + [5, 3]
# Tilted cluster
angle = np.pi / 4
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
c3 = (np.random.randn(n_per, 2) @ np.diag([2, 0.4]) @ R.T) + [-3, 4]

X_aniso = np.vstack([c1, c2, c3])
y_aniso = np.array([0]*n_per + [1]*n_per + [2]*n_per)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Ground truth
for k in range(3):
    mask = y_aniso == k
    axes[0].scatter(X_aniso[mask, 0], X_aniso[mask, 1], c=colors[k], alpha=0.5, s=15)
axes[0].set_title('Ground Truth')
axes[0].grid(True, alpha=0.3)

# K-means
from sklearn.cluster import KMeans as SkKMeans
km_aniso = SkKMeans(n_clusters=3, random_state=42, n_init=10).fit(X_aniso)
for k in range(3):
    mask = km_aniso.labels_ == k
    axes[1].scatter(X_aniso[mask, 0], X_aniso[mask, 1], c=colors[k], alpha=0.5, s=15)
axes[1].scatter(km_aniso.cluster_centers_[:, 0], km_aniso.cluster_centers_[:, 1],
                c='black', marker='X', s=200, edgecolors='white', linewidths=2)
axes[1].set_title('K-Means (assumes spherical clusters)')
axes[1].grid(True, alpha=0.3)

# GMM
gmm_aniso = GaussianMixtureModel(n_components=3, random_state=42)
gmm_aniso.fit(X_aniso)
for k in range(3):
    mask = gmm_aniso.labels_ == k
    axes[2].scatter(X_aniso[mask, 0], X_aniso[mask, 1], c=colors[k], alpha=0.5, s=15)
    draw_ellipse(gmm_aniso.means_[k], gmm_aniso.covariances_[k], axes[2], n_std=2,
                 fill=False, edgecolor=colors[k], linewidth=2)
axes[2].set_title('GMM (captures elliptical shape)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('K-Means vs GMM on Anisotropic Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Selection with BIC/AIC

In [ ]:
K_range = range(1, 8)
bics = []
aics = []

for k in K_range:
    gmm_temp = GaussianMixtureModel(n_components=k, random_state=42)
    gmm_temp.fit(X_gmm)
    bics.append(gmm_temp.bic(X_gmm))
    aics.append(gmm_temp.aic(X_gmm))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_range), bics, 'bo-', linewidth=2, label='BIC', markersize=8)
ax.plot(list(K_range), aics, 'rs-', linewidth=2, label='AIC', markersize=8)
ax.axvline(x=3, color='green', linestyle='--', alpha=0.5, label='True K=3')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Information Criterion')
ax.set_title('Model Selection: BIC and AIC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best K by BIC: {list(K_range)[np.argmin(bics)]}")
print(f"Best K by AIC: {list(K_range)[np.argmin(aics)]}")

## 7. Density Estimation with GMM

Unlike K-means, GMM gives a proper probability density that can be used for density estimation, anomaly detection, and data generation.

In [ ]:
# Fit GMM and plot density contours
gmm_density = GaussianMixtureModel(n_components=3, random_state=42)
gmm_density.fit(X_gmm)

# Create grid
h = 0.2
x_min, x_max = X_gmm[:, 0].min() - 3, X_gmm[:, 0].max() + 3
y_min, y_max = X_gmm[:, 1].min() - 3, X_gmm[:, 1].max() + 3
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

# Compute density
density = np.zeros(grid.shape[0])
for k in range(gmm_density.n_components):
    density += gmm_density.weights_[k] * multivariate_gaussian_pdf(
        grid, gmm_density.means_[k], gmm_density.covariances_[k]
    )
density = density.reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot
ax = axes[0]
cs = ax.contourf(xx, yy, density, levels=30, cmap='viridis', alpha=0.8)
ax.scatter(X_gmm[:, 0], X_gmm[:, 1], c='white', alpha=0.3, s=5)
plt.colorbar(cs, ax=ax, label='Density')
ax.set_title('GMM Density Estimation')
ax.grid(True, alpha=0.2)

# 3D surface
ax3d = fig.add_subplot(122, projection='3d')
ax3d.plot_surface(xx, yy, density, cmap='viridis', alpha=0.7)
ax3d.set_title('Density Surface')
ax3d.set_xlabel('X1')
ax3d.set_ylabel('X2')
ax3d.set_zlabel('Density')
axes[1].set_visible(False)

plt.tight_layout()
plt.show()

## 8. Local Optima: Multiple EM Runs

In [ ]:
# Show that EM can get stuck in different local optima
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

final_lls = []
for trial, seed in enumerate(range(8)):
    gmm_temp = GaussianMixtureModel(n_components=3, random_state=seed * 7 + 1)
    gmm_temp.fit(X_gmm)
    final_lls.append(gmm_temp.log_likelihoods_[-1])
    
    ax = axes[trial // 4, trial % 4]
    for k in range(3):
        mask = gmm_temp.labels_ == k
        ax.scatter(X_gmm[mask, 0], X_gmm[mask, 1], c=colors[k % 3], alpha=0.4, s=10)
        draw_ellipse(gmm_temp.means_[k], gmm_temp.covariances_[k], ax, n_std=2,
                     fill=False, edgecolor=colors[k % 3], linewidth=1.5)
    ax.set_title(f'Seed {seed}: LL={gmm_temp.log_likelihoods_[-1]:.0f}', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('EM with Different Initializations (Local Optima)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Log-likelihoods: {[f'{ll:.0f}' for ll in final_lls]}")
print(f"Best: {max(final_lls):.0f}, Worst: {min(final_lls):.0f}")

## 9. Compare with sklearn

In [ ]:
from sklearn.mixture import GaussianMixture as SklearnGMM
from sklearn.metrics import adjusted_rand_score

our_gmm = GaussianMixtureModel(n_components=3, random_state=42)
our_gmm.fit(X_gmm)

sk_gmm = SklearnGMM(n_components=3, random_state=42, n_init=1)
sk_gmm.fit(X_gmm)

print("Our GMM:")
print(f"  ARI vs true: {adjusted_rand_score(y_gmm, our_gmm.labels_):.4f}")
print(f"  BIC: {our_gmm.bic(X_gmm):.2f}")

print(f"\nsklearn GMM:")
print(f"  ARI vs true: {adjusted_rand_score(y_gmm, sk_gmm.predict(X_gmm)):.4f}")
print(f"  BIC: {sk_gmm.bic(X_gmm):.2f}")

print(f"\nWeight comparison:")
print(f"  Ours:    {np.sort(our_gmm.weights_)[::-1]}")
print(f"  sklearn: {np.sort(sk_gmm.weights_)[::-1]}")

---

## Interview Questions

### "What is EM doing intuitively?"
EM alternates between two steps to maximize likelihood when there are latent (hidden) variables:
- **E-step:** Given current parameters, compute the expected value of latent variables ("what cluster is each point likely from?")
- **M-step:** Given these soft assignments, update parameters to maximize likelihood ("given assignments, what are the best cluster parameters?")

It's **coordinate ascent on the ELBO** (Evidence Lower BOund): E-step optimizes the variational distribution, M-step optimizes the model parameters.

### "Can EM get stuck in local optima?"
Yes! The log-likelihood surface is non-convex for mixtures. EM is guaranteed to increase the log-likelihood at each step, but may converge to a local maximum. Mitigation: run multiple times with different initializations, use K-means++ for init.

### "E-step vs M-step?"
- E-step: Inference (compute posteriors over latent variables given current parameters)
- M-step: Learning (optimize parameters given current posteriors)
- Together: iterative algorithm that is guaranteed to converge

### "Connection to variational inference?"
EM is a special case of variational inference where the variational family is the exact posterior (no approximation gap). In general VI, we use a restricted family $q(z)$ and maximize ELBO = $\mathbb{E}_q[\log p(x,z)] - \mathbb{E}_q[\log q(z)]$. EM does this when q is the exact posterior.

### Notes
- **EM is general:** Not just for GMM. Used for HMMs (Baum-Welch), factor analysis, missing data, and latent variable models.
- **Singularity problem:** If a Gaussian component collapses to a single point, its variance goes to 0 and likelihood goes to infinity. Fix with regularization ($\Sigma + \epsilon I$) or minimum variance constraints.
- **ELBO connection:** $\log p(X|\theta) = \text{ELBO}(q, \theta) + \text{KL}(q \| p(Z|X,\theta))$. E-step sets KL=0, M-step maximizes ELBO w.r.t. $\theta$.

In [ ]:
print("Notebook 12 complete: GMM & EM from Scratch")
print("="*50)
print("Implemented:")
print("  - Multivariate Gaussian PDF")
print("  - Full GMM with E-step and M-step")
print("  - Log-likelihood tracking")
print("  - BIC/AIC model selection")
print("  - Density estimation")
print("  - K-means vs GMM comparison")
print("  - Local optima demonstration")
print("  - Validated against sklearn")